# 06. Selective Classification & Risk-Coverage Evaluation (E7)
**Role 2: Trust Engine Engineer**  
**Handbook Reference:** Section 1.2, Section 5.8 (Selective Classification), Section 7.3 (Stages 2–5), Section 7.6 (E7), Section 7.10  
**Evaluated Model:** Production Calibrated Classifier (`backend/models/crop_pipeline_calibrated.joblib` — Extra Trees + Isotonic Calibration, 5-fold CV)

---

### Methodological Context & Scope
Selective classification allows a machine learning system to abstain from making high-risk recommendations, trading **coverage** (fraction of inputs answered) for **reliability** (reduced error among answered inputs). 

This notebook integrates and evaluates the complete 5-stage Trust Engine pipeline across three risk profiles (**Conservative**, **Balanced**, **Bold**):
1. **Clean In-Distribution Validation Data:** All 330 genuine validation inputs (`data/splits/val_idx.npy`).
2. **Physically Perturbed Known Inputs:** 5 independent physical noise draws (seeds 1001–1005) under approved primary 1-sigma noise (1,650 inputs total).
3. **Manufactured Out-of-Distribution Unknowns:** 1,000 Box-Uniform inputs (seed 2020) and 1,000 Shuffled-Column inputs (seed 4040).

### Strict Methodological Safeguards Enforced:
1. **Zero Data Leakage:** Evaluated exclusively on `val_idx.npy`. The test set (`test_idx.npy`) remains 100% untouched.
2. **Production Calibrated Model Exclusively:** Loaded via `TrustEngine.load("backend/models")`, loading `crop_pipeline_calibrated.joblib`. Zero uncalibrated baseline fallback.
3. **Equivalence Verification:** Verifies 180 comparisons (60 rows × 3 profiles) confirming that cached per-row Stage 3/4 evaluation is 100% bitwise equivalent to direct unpatched `engine.assess()`.
4. **Objective Empirical Reporting:** No thresholds are tuned. Targets are observational ("none observed" rather than "proven"). Overlapping profile confidence intervals are explicitly documented.
5. **Clean Standalone Execution:** Self-contained execution without reading scratch files.

In [1]:
import os
import sys
import json
import time
from pathlib import Path
from typing import Any, Dict, List, Tuple
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Robust root path resolution for notebook execution
REPO_ROOT = Path("..").resolve() if Path("..", "backend").exists() else Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from backend.noise import apply_noise
from backend.trust_engine import TrustEngine

# Canonical feature ordering
CANONICAL_FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]

# Figures output directory
FIGURES_DIR = REPO_ROOT / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working Directory: {Path.cwd()}")
print(f"Resolved REPO_ROOT: {REPO_ROOT}")
print(f"Figures Destination: {FIGURES_DIR}")


Working Directory: C:\Users\varsh\OneDrive\Desktop\crop-trust-engine\notebooks
Resolved REPO_ROOT: C:\Users\varsh\OneDrive\Desktop\crop-trust-engine
Figures Destination: C:\Users\varsh\OneDrive\Desktop\crop-trust-engine\docs\figures


---
## Part 1: Dataset Loading, Split Hygiene, and Unknown Generators

We load `Crop_recommendation.csv`, training split indices (`train_idx.npy`), and validation split indices (`val_idx.npy`).  
The test set (`test_idx.npy`) is checked for non-overlap and strictly excluded from memory.

In [2]:
def load_dataset_and_splits():
    data_path = REPO_ROOT / "data" / "Crop_recommendation.csv"
    df = pd.read_csv(data_path)
    
    split_dir = REPO_ROOT / "data" / "splits"
    train_idx = np.load(split_dir / "train_idx.npy")
    val_idx = np.load(split_dir / "val_idx.npy")
    
    # Verify split hygiene
    assert set(train_idx).isdisjoint(set(val_idx)), "ERROR: train and val overlap!"
    assert (split_dir / "test_idx.npy").exists(), "test_idx.npy missing!"
    
    X_train = df.iloc[train_idx][CANONICAL_FEATURES].values
    y_train = df.iloc[train_idx]["label"].values
    X_val = df.iloc[val_idx][CANONICAL_FEATURES].values
    y_val = df.iloc[val_idx]["label"].values
    
    classes = sorted(list(np.unique(y_train)))
    train_min = X_train.min(axis=0)
    train_max = X_train.max(axis=0)
    
    return df, train_idx, val_idx, X_train, y_train, X_val, y_val, classes, train_min, train_max

def generate_box_uniform(n: int, bounds_min: np.ndarray, bounds_max: np.ndarray, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return rng.uniform(bounds_min, bounds_max, size=(n, len(bounds_min)))

def generate_shuffled_columns(X_source: np.ndarray, n: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X_source), size=n, replace=True)
    sampled = X_source[idx].copy()
    for col in range(sampled.shape[1]):
        rng.shuffle(sampled[:, col])
    return sampled

def get_val_eval_subset(df: pd.DataFrame, val_idx: np.ndarray) -> np.ndarray:
    SEED = 42
    rng_val = np.random.default_rng(seed=SEED)
    y_val = df.iloc[val_idx]["label"].values
    classes = sorted(list(np.unique(y_val)))
    eval_idx_list = []
    for crop in classes:
        crop_pos = np.where(y_val == crop)[0]
        shuffled = rng_val.permutation(crop_pos)
        eval_idx_list.extend(shuffled[10:])  # 5 per crop
    eval_sub_idx = np.array(eval_idx_list)
    return df.iloc[val_idx[eval_sub_idx]][CANONICAL_FEATURES].values

df, train_idx, val_idx, X_train, y_train, X_val, y_val, classes, train_min, train_max = load_dataset_and_splits()
X_val_eval = get_val_eval_subset(df, val_idx)

print(f"Loaded {len(df)} total rows.")
print(f"Train split: {len(X_train)} rows | Validation split: {len(X_val)} rows (test split excluded).")
print(f"Held-out val_eval partition: {len(X_val_eval)} rows across {len(classes)} crops.")


Loaded 2200 total rows.
Train split: 1540 rows | Validation split: 330 rows (test split excluded).
Held-out val_eval partition: 110 rows across 22 crops.


---
## Part 2: Evaluation Architecture & Exact Equivalence Check

In `TrustEngine.assess()`, Stage 3 (calibrated probabilities) and Stage 4 (Monte Carlo stability with $n=200$ samples) depend only on the input features $x$, not the risk profile.

To evaluate $N = 3,980$ rows across 3 risk profiles without redundant Monte Carlo recalculations:
1. We compute Stage 2 OOD distance, Stage 3 probabilities (`assess_probabilities`), and Stage 4 stability (`assess_stability`) once per row using the real production methods.
2. We simulate each profile verdict via `engine.assess(row, risk=profile)` using `patch.object` on `assess_probabilities` and `assess_stability` with the exact cached per-row values.
3. We run an **Equivalence Check on 60 representative rows × 3 profiles = 180 comparisons** against unpatched direct `assess()`, asserting 100% bitwise identity.

In [3]:
# Load production engine
engine = TrustEngine.load(REPO_ROOT / "backend" / "models")
print("Loaded production TrustEngine with calibrated pipeline.")

def evaluate_single_dataset(name: str, X_arr: np.ndarray, y_true: np.ndarray, engine: TrustEngine) -> Dict[str, Any]:
    t0 = time.time()
    n_rows = len(X_arr)
    
    stage2_results = []
    cached_prob_res = []
    cached_stab_res = []
    top_predicted_crops = []
    top_predicted_probs = []
    margins = []
    stabilities = []
    
    for i in range(n_rows):
        row_dict = {feat: float(X_arr[i, j]) for j, feat in enumerate(CANONICAL_FEATURES)}
        
        # Stage 2: OOD distance
        ood_res = engine.assess_ood(row_dict, risk="balanced")
        stage2_results.append(ood_res)
        
        # Stage 3 & 4 only if passing Stage 2 under Bold (most permissive)
        min_d2 = ood_res["min_distance"]
        if min_d2 <= 13.75:
            # Real production per-row Stage 3 & 4
            prob_res = engine.assess_probabilities(row_dict)
            stab_res = engine.assess_stability(row_dict, prob_res["top_crop"], n_samples=200, seed=42)
            
            cached_prob_res.append(prob_res)
            cached_stab_res.append(stab_res)
            top_predicted_crops.append(prob_res["top_crop"])
            top_predicted_probs.append(prob_res["top_probability"])
            margins.append(prob_res["margin"])
            stabilities.append(stab_res["stability"])
        else:
            cached_prob_res.append(None)
            cached_stab_res.append(None)
            top_predicted_crops.append(None)
            top_predicted_probs.append(None)
            margins.append(None)
            stabilities.append(None)
            
    elapsed = time.time() - t0
    return {
        "name": name,
        "X_arr": X_arr,
        "y_true": y_true,
        "stage2_results": stage2_results,
        "cached_prob_res": cached_prob_res,
        "cached_stab_res": cached_stab_res,
        "top_predicted_crops": top_predicted_crops,
        "top_predicted_probs": top_predicted_probs,
        "margins": margins,
        "stabilities": stabilities,
        "elapsed_sec": elapsed,
    }

def compute_profile_verdicts(dataset_res: Dict[str, Any], engine: TrustEngine, profile: str) -> List[Dict[str, Any]]:
    X_arr = dataset_res["X_arr"]
    cached_prob = dataset_res["cached_prob_res"]
    cached_stab = dataset_res["cached_stab_res"]
    n_rows = len(X_arr)
    verdicts = []
    
    for i in range(n_rows):
        row_dict = {feat: float(X_arr[i, j]) for j, feat in enumerate(CANONICAL_FEATURES)}
        prob_res = cached_prob[i]
        stab_res = cached_stab[i]
        
        if prob_res is None or stab_res is None:
            # Definite Stage 2 refusal
            res = engine.assess(row_dict, risk=profile)
        else:
            with patch.object(engine, "assess_probabilities", return_value=prob_res), \
                 patch.object(engine, "assess_stability", return_value=stab_res):
                res = engine.assess(row_dict, risk=profile)
        verdicts.append(res)
    return verdicts

def run_equivalence_check(engine: TrustEngine, eval_clean: Dict[str, Any], eval_noisy_list: List[Dict[str, Any]], eval_box: Dict[str, Any], eval_shuf: Dict[str, Any]) -> Tuple[bool, int, str]:
    profiles = ["conservative", "balanced", "bold"]
    clean_rows = [(eval_clean["X_arr"][i], eval_clean["cached_prob_res"][i], eval_clean["cached_stab_res"][i], f"clean_{i}") for i in range(20)]
    
    noisy_rows = []
    for s_idx, noisy_ds in enumerate(eval_noisy_list):
        for r_idx in range(4):
            noisy_rows.append((noisy_ds["X_arr"][r_idx], noisy_ds["cached_prob_res"][r_idx], noisy_ds["cached_stab_res"][r_idx], f"noisy_{s_idx}_{r_idx}"))
            
    unknown_passing = []
    for ds in [eval_shuf, eval_box]:
        for i in range(len(ds["X_arr"])):
            if ds["cached_prob_res"][i] is not None:
                unknown_passing.append((ds["X_arr"][i], ds["cached_prob_res"][i], ds["cached_stab_res"][i], f"unknown_{ds['name']}_{i}"))
            if len(unknown_passing) == 20:
                break
        if len(unknown_passing) == 20:
            break
            
    test_pool = clean_rows[:20] + noisy_rows[:20] + unknown_passing[:20]
    assert len(test_pool) == 60, f"Expected 60 test rows, got {len(test_pool)}"
    
    count = 0
    for x_arr, prob_res, stab_res, label in test_pool:
        row_dict = {f: float(x_arr[j]) for j, f in enumerate(CANONICAL_FEATURES)}
        for p in profiles:
            count += 1
            direct_res = engine.assess(row_dict, risk=p)
            if prob_res is None or stab_res is None:
                patched_res = engine.assess(row_dict, risk=p)
            else:
                with patch.object(engine, "assess_probabilities", return_value=prob_res), \
                     patch.object(engine, "assess_stability", return_value=stab_res):
                    patched_res = engine.assess(row_dict, risk=p)
            if direct_res != patched_res:
                return False, count, f"Mismatch at row {label}, profile {p}"
    return True, count, "All 180 comparisons bitwise equal"

print("Evaluation pipeline initialized.")


C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeClassifier from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreesClassifier from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpic

Loaded production TrustEngine with calibrated pipeline.
Evaluation pipeline initialized.


C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator CalibratedClassifierCV from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


---
## Part 3: Dataset Evaluation Execution

We execute Stage 2–5 evaluation on:
1. `eval_clean`: 330 clean validation rows.
2. `eval_noisy_list`: 5 independent noisy realizations (seeds 1001–1005, 330 rows each = 1,650 rows).
3. `eval_box_main`: 1,000 Box-Uniform unknown rows (seed 2020).
4. `eval_shuf_main`: 1,000 Shuffled-Column unknown rows (seed 4040).
Then we verify equivalence against unpatched `engine.assess()`.

In [4]:
# 1. Clean validation
t0 = time.time()
print("Evaluating clean validation rows (330 inputs)...")
eval_clean = evaluate_single_dataset("clean_val", X_val, y_val, engine)
print(f"Clean evaluation completed in {eval_clean['elapsed_sec']:.2f}s.")

# 2. Noisy validation copies (seeds 1001 to 1005)
noisy_seeds = [1001, 1002, 1003, 1004, 1005]
eval_noisy_list = []
print(f"Evaluating noisy validation copies across seeds {noisy_seeds} (1,650 inputs total)...")
for s in noisy_seeds:
    X_noisy = apply_noise(X_val, profile="primary_1sigma", seed=s)
    res_s = evaluate_single_dataset(f"noisy_seed_{s}", X_noisy, y_val, engine)
    eval_noisy_list.append(res_s)
    print(f"  Seed {s}: completed in {res_s['elapsed_sec']:.2f}s.")

# 3. Unknowns (Main Benchmark)
print("Evaluating Box-Uniform unknowns (seed 2020, 1000 inputs)...")
box_2020 = generate_box_uniform(1000, train_min, train_max, seed=2020)
eval_box_main = evaluate_single_dataset("box_seed_2020", box_2020, np.full(1000, "UNKNOWN"), engine)

print("Evaluating Shuffled-Column unknowns (seed 4040, 1000 inputs)...")
shuf_4040 = generate_shuffled_columns(X_val_eval, 1000, seed=4040)
eval_shuf_main = evaluate_single_dataset("shuf_seed_4040", shuf_4040, np.full(1000, "UNKNOWN"), engine)

# 4. Equivalence Check (180 comparisons)
print("\nRunning Equivalence Verification...")
equiv_ok, equiv_count, equiv_msg = run_equivalence_check(engine, eval_clean, eval_noisy_list, eval_box_main, eval_shuf_main)
assert equiv_ok, f"Equivalence check failed: {equiv_msg}"
print(f"Equivalence Verification Result: {equiv_msg} ({equiv_count}/180 comparisons MATCHED).")


Evaluating clean validation rows (330 inputs)...


Clean evaluation completed in 45.08s.
Evaluating noisy validation copies across seeds [1001, 1002, 1003, 1004, 1005] (1,650 inputs total)...


  Seed 1001: completed in 32.37s.


  Seed 1002: completed in 31.13s.


  Seed 1003: completed in 28.12s.


  Seed 1004: completed in 26.59s.


  Seed 1005: completed in 25.59s.
Evaluating Box-Uniform unknowns (seed 2020, 1000 inputs)...


Evaluating Shuffled-Column unknowns (seed 4040, 1000 inputs)...



Running Equivalence Verification...


Equivalence Verification Result: All 180 comparisons bitwise equal (180/180 comparisons MATCHED).


---
## Section A: Risk-Coverage Sweep & Profile Operating Points

We sweep threshold $t \in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 1.00]$ using the score:
$$\text{score} = \min(\text{top\_probability}, \text{stability})$$
evaluated on inputs that pass the OOD gate ($d^2 \le 12.92$).
- **Coverage Denominator:** Total rows in dataset ($N = 330$ clean, $N = 1,650$ noisy pooled).
- **Selective Risk:** Error rate on recommended inputs ($\text{Wrong} / \text{Recommended}$).

Figures are saved to `docs/figures/risk_coverage_clean.png` and `docs/figures/risk_coverage_noisy.png`.

In [5]:
def compute_sweep(dataset_list: List[Dict[str, Any]], total_n: int, name: str) -> pd.DataFrame:
    # Pool passing rows
    pooled_min_d2 = []
    pooled_probs = []
    pooled_stabs = []
    pooled_preds = []
    pooled_true = []
    
    for ds in dataset_list:
        y_t = ds["y_true"]
        for i in range(len(ds["X_arr"])):
            pooled_min_d2.append(ds["stage2_results"][i]["min_distance"])
            p_res = ds["cached_prob_res"][i]
            s_res = ds["cached_stab_res"][i]
            if p_res is not None and s_res is not None:
                pooled_probs.append(p_res["top_probability"])
                pooled_stabs.append(s_res["stability"])
                pooled_preds.append(p_res["top_crop"])
            else:
                pooled_probs.append(0.0)
                pooled_stabs.append(0.0)
                pooled_preds.append("REFUSED")
            pooled_true.append(y_t[i])
            
    min_d2 = np.array(pooled_min_d2)
    probs = np.array(pooled_probs)
    stabs = np.array(pooled_stabs)
    scores = np.minimum(probs, stabs)
    preds = np.array(pooled_preds)
    trues = np.array(pooled_true)
    
    thresholds_t = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 1.00]
    records = []
    for t in thresholds_t:
        mask = (min_d2 <= 12.92) & (scores >= t)
        n_rec = int(np.sum(mask))
        cov = n_rec / total_n
        if n_rec > 0:
            n_wrong = int(np.sum(preds[mask] != trues[mask]))
            risk = n_wrong / n_rec
        else:
            n_wrong = 0
            risk = 0.0
        records.append({
            "threshold_t": t,
            "n_recommended": n_rec,
            "total_instances": total_n,
            "coverage": round(cov, 4),
            "n_wrong_recommended": n_wrong,
            "selective_risk": round(risk, 4)
        })
    return pd.DataFrame(records)

df_sweep_clean = compute_sweep([eval_clean], 330, "Clean")
df_sweep_noisy = compute_sweep(eval_noisy_list, 1650, "Noisy Pooled")

print("=== RISK-COVERAGE SWEEP: CLEAN VALIDATION (N=330) ===")
print(df_sweep_clean.to_string(index=False))

print("\n=== RISK-COVERAGE SWEEP: NOISY POOLED VALIDATION (N=1650) ===")
print(df_sweep_noisy.to_string(index=False))

# Plotting Risk-Coverage Curves
def plot_risk_coverage_curve(df_sweep, operating_pts, title, out_path):
    fig, ax1 = plt.subplots(figsize=(8, 5))
    color_cov = "#1f77b4"
    color_risk = "#d62728"
    
    ax1.set_xlabel("Confidence/Stability Threshold (t)", fontsize=11, fontweight="bold")
    ax1.set_ylabel("Coverage (Fraction Recommended)", color=color_cov, fontsize=11, fontweight="bold")
    l1 = ax1.plot(df_sweep["threshold_t"], df_sweep["coverage"] * 100, color=color_cov, marker="o", lw=2, label="Coverage (%)")
    ax1.tick_params(axis="y", labelcolor=color_cov)
    ax1.set_ylim(-2, 105)
    
    ax2 = ax1.twinx()
    ax2.set_ylabel("Selective Risk (Error Rate among Recommended %)", color=color_risk, fontsize=11, fontweight="bold")
    l2 = ax2.plot(df_sweep["threshold_t"], df_sweep["selective_risk"] * 100, color=color_risk, marker="s", lw=2, linestyle="--", label="Selective Risk (%)")
    ax2.tick_params(axis="y", labelcolor=color_risk)
    
    # Mark operating points
    markers = {"Conservative": "^", "Balanced": "D", "Bold": "v"}
    for p_name, pt in operating_pts.items():
        ax1.scatter([pt["t_eff"]], [pt["coverage"] * 100], color="#2ca02c", s=90, marker=markers[p_name], zorder=5)
        ax1.annotate(f"{p_name}\n({pt['coverage']*100:.1f}%)", (pt["t_eff"], pt["coverage"] * 100), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=8, fontweight="bold")
        
    plt.title(title, fontsize=12, fontweight="bold", pad=12)
    fig.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()
    print(f"Saved figure: {out_path}")

# Operating points
op_clean = {
    "Conservative": {"t_eff": 0.90, "coverage": 251 / 330, "risk": 0.0},
    "Balanced": {"t_eff": 0.80, "coverage": 288 / 330, "risk": 0.0},
    "Bold": {"t_eff": 0.65, "coverage": 315 / 330, "risk": 0.0},
}
op_noisy = {
    "Conservative": {"t_eff": 0.90, "coverage": 517 / 1650, "risk": 4 / 517},
    "Balanced": {"t_eff": 0.80, "coverage": 701 / 1650, "risk": 13 / 701},
    "Bold": {"t_eff": 0.65, "coverage": 858 / 1650, "risk": 28 / 858},
}

plot_risk_coverage_curve(df_sweep_clean, op_clean, "E7: Risk-Coverage Curve (Clean Validation, N=330)", FIGURES_DIR / "risk_coverage_clean.png")
plot_risk_coverage_curve(df_sweep_noisy, op_noisy, "E7: Risk-Coverage Curve (Noisy Pooled, 5 Seeds, N=1650)", FIGURES_DIR / "risk_coverage_noisy.png")


=== RISK-COVERAGE SWEEP: CLEAN VALIDATION (N=330) ===
 threshold_t  n_recommended  total_instances  coverage  n_wrong_recommended  selective_risk
        0.50            324              330    0.9818                    1          0.0031
        0.60            320              330    0.9697                    1          0.0031
        0.70            311              330    0.9424                    0          0.0000
        0.80            293              330    0.8879                    0          0.0000
        0.90            263              330    0.7970                    0          0.0000
        0.95            232              330    0.7030                    0          0.0000
        1.00            118              330    0.3576                    0          0.0000

=== RISK-COVERAGE SWEEP: NOISY POOLED VALIDATION (N=1650) ===
 threshold_t  n_recommended  total_instances  coverage  n_wrong_recommended  selective_risk
        0.50           1003             1650    0.6079 

Saved figure: C:\Users\varsh\OneDrive\Desktop\crop-trust-engine\docs\figures\risk_coverage_clean.png


Saved figure: C:\Users\varsh\OneDrive\Desktop\crop-trust-engine\docs\figures\risk_coverage_noisy.png


---
## Section B: Error Attribution Breakdown

We trace error flow across 5 categories:
1. **Plain Model (All Rows):** Model top crop error rate without Trust Engine filtering.
2. **Passed OOD Gate:** Inputs not refused at Stage 2.
3. **Refused Rows:** Inputs flagged OOD (reports counterfactual plain model error).
4. **Caution Rows:** Inputs routed to Stage 5 caution.
5. **Recommend Rows:** Inputs reaching final recommendation.

In [6]:
def compute_attribution_for_dataset(ds_list: List[Dict[str, Any]], engine: TrustEngine) -> Dict[str, Any]:
    all_trues = []
    all_plain_preds = []
    
    for ds in ds_list:
        X = ds["X_arr"]
        df_x = pd.DataFrame(X, columns=CANONICAL_FEATURES)
        probs = engine.pipeline.predict_proba(df_x)
        preds = [engine.classes[idx] for idx in np.argmax(probs, axis=1)]
        all_plain_preds.extend(preds)
        all_trues.extend(ds["y_true"])
        
    all_trues = np.array(all_trues)
    all_plain_preds = np.array(all_plain_preds)
    plain_wrong = int(np.sum(all_plain_preds != all_trues))
    n_total = len(all_trues)
    
    profiles = ["conservative", "balanced", "bold"]
    by_profile = {}
    
    for p in profiles:
        verdicts = []
        for ds in ds_list:
            v_ds = compute_profile_verdicts(ds, engine, p)
            verdicts.extend(v_ds)
            
        not_ref_idx = [i for i, v in enumerate(verdicts) if v["verdict"] != "refuse"]
        ref_idx = [i for i, v in enumerate(verdicts) if v["verdict"] == "refuse"]
        caut_idx = [i for i, v in enumerate(verdicts) if v["verdict"] == "caution"]
        rec_idx = [i for i, v in enumerate(verdicts) if v["verdict"] == "recommend"]
        
        def get_stat(idx_list):
            cnt = len(idx_list)
            if cnt == 0:
                return {"n_rows": 0, "wrong": 0, "error_rate": 0.0}
            wr = int(np.sum(all_plain_preds[idx_list] != all_trues[idx_list]))
            return {"n_rows": cnt, "wrong": wr, "error_rate": round(wr / cnt, 4)}
            
        by_profile[p] = {
            "not_refused": get_stat(not_ref_idx),
            "refused": get_stat(ref_idx),
            "caution": get_stat(caut_idx),
            "recommend": get_stat(rec_idx),
        }
        
    return {
        "plain_model": {"n_rows": n_total, "wrong": plain_wrong, "error_rate": round(plain_wrong / n_total, 4)},
        "by_profile": by_profile
    }

attr_clean = compute_attribution_for_dataset([eval_clean], engine)
attr_noisy_pooled = compute_attribution_for_dataset(eval_noisy_list, engine)

print("=== ATTRIBUTION: CLEAN VALIDATION (N=330) ===")
print(f"Plain Model: {attr_clean['plain_model']['wrong']} wrong of {attr_clean['plain_model']['n_rows']} ({attr_clean['plain_model']['error_rate']*100:.2f}%)")
for p in ["conservative", "balanced", "bold"]:
    bp = attr_clean["by_profile"][p]
    print(f"[{p.upper()}] Not Ref: {bp['not_refused']['n_rows']} (wr={bp['not_refused']['wrong']}) | Ref: {bp['refused']['n_rows']} (wr={bp['refused']['wrong']}) | Caut: {bp['caution']['n_rows']} (wr={bp['caution']['wrong']}) | Rec: {bp['recommend']['n_rows']} (wr={bp['recommend']['wrong']})")

print("\n=== ATTRIBUTION: NOISY POOLED (N=1650) ===")
print(f"Plain Model: {attr_noisy_pooled['plain_model']['wrong']} wrong of {attr_noisy_pooled['plain_model']['n_rows']} ({attr_noisy_pooled['plain_model']['error_rate']*100:.2f}%)")
for p in ["conservative", "balanced", "bold"]:
    bp = attr_noisy_pooled["by_profile"][p]
    print(f"[{p.upper()}] Not Ref: {bp['not_refused']['n_rows']} (wr={bp['not_refused']['wrong']}) | Ref: {bp['refused']['n_rows']} (wr={bp['refused']['wrong']}) | Caut: {bp['caution']['n_rows']} (wr={bp['caution']['wrong']}) | Rec: {bp['recommend']['n_rows']} (wr={bp['recommend']['wrong']})")

# Breakdown by individual noisy seed
print("\n=== ATTRIBUTION: INDIVIDUAL NOISY SEEDS (330 rows each) ===")
for s_idx, s in enumerate(noisy_seeds):
    attr_s = compute_attribution_for_dataset([eval_noisy_list[s_idx]], engine)
    print(f"Seed {s}: Plain wrong = {attr_s['plain_model']['wrong']}/330 ({attr_s['plain_model']['error_rate']*100:.2f}%)")
    for p in ["conservative", "balanced", "bold"]:
        rec = attr_s["by_profile"][p]["recommend"]
        print(f"  {p[:4]}: Rec={rec['n_rows']} (wrong={rec['wrong']})")


=== ATTRIBUTION: CLEAN VALIDATION (N=330) ===
Plain Model: 2 wrong of 330 (0.61%)
[CONSERVATIVE] Not Ref: 324 (wr=2) | Ref: 6 (wr=0) | Caut: 73 (wr=2) | Rec: 251 (wr=0)
[BALANCED] Not Ref: 326 (wr=2) | Ref: 4 (wr=0) | Caut: 38 (wr=2) | Rec: 288 (wr=0)
[BOLD] Not Ref: 327 (wr=2) | Ref: 3 (wr=0) | Caut: 12 (wr=2) | Rec: 315 (wr=0)

=== ATTRIBUTION: NOISY POOLED (N=1650) ===
Plain Model: 97 wrong of 1650 (5.88%)
[CONSERVATIVE] Not Ref: 949 (wr=51) | Ref: 701 (wr=46) | Caut: 432 (wr=47) | Rec: 517 (wr=4)
[BALANCED] Not Ref: 1045 (wr=56) | Ref: 605 (wr=41) | Caut: 344 (wr=43) | Rec: 701 (wr=13)
[BOLD] Not Ref: 1102 (wr=59) | Ref: 548 (wr=38) | Caut: 244 (wr=31) | Rec: 858 (wr=28)

=== ATTRIBUTION: INDIVIDUAL NOISY SEEDS (330 rows each) ===


Seed 1001: Plain wrong = 13/330 (3.94%)
  cons: Rec=103 (wrong=0)
  bala: Rec=138 (wrong=3)
  bold: Rec=167 (wrong=3)


Seed 1002: Plain wrong = 22/330 (6.67%)
  cons: Rec=99 (wrong=0)
  bala: Rec=139 (wrong=1)
  bold: Rec=173 (wrong=5)


Seed 1003: Plain wrong = 25/330 (7.58%)
  cons: Rec=115 (wrong=2)
  bala: Rec=144 (wrong=5)
  bold: Rec=173 (wrong=8)


Seed 1004: Plain wrong = 16/330 (4.85%)
  cons: Rec=100 (wrong=2)
  bala: Rec=143 (wrong=2)
  bold: Rec=172 (wrong=8)


Seed 1005: Plain wrong = 21/330 (6.36%)
  cons: Rec=100 (wrong=0)
  bala: Rec=137 (wrong=2)
  bold: Rec=173 (wrong=4)


---
## Section C: Why Noisy Rows Are Refused

We examine:
1. Within-crop training dispersion ($\sigma_{\text{within}}$) versus applied noise.
2. Single-feature perturbation tests on the clean validation set.
3. Crop-by-crop refusal rates under full physical noise.

In [7]:
# 1. Feature Within-Crop SD vs Applied 1-Sigma Noise
feature_stds = []
with open(REPO_ROOT / "backend" / "models" / "noise_model.json", "r") as f:
    noise_cfg = json.load(f)["profiles"]["primary_1sigma"]["features"]

for feat in CANONICAL_FEATURES:
    stds_per_crop = []
    for c in classes:
        mask = (df.iloc[train_idx]["label"] == c)
        stds_per_crop.append(float(df.iloc[train_idx][mask][feat].std()))
    m_std = float(np.mean(stds_per_crop))
    min_std = float(np.min(stds_per_crop))
    max_std = float(np.max(stds_per_crop))
    
    cfg = noise_cfg[feat]
    if cfg["scale_type"] == "relative":
        typical_val = float(df.iloc[train_idx][feat].mean())
        typ_noise = max(cfg["sigma_floor"], cfg["relative_ratio"] * typical_val)
        formula = f"rel {int(cfg['relative_ratio']*100)}%, floor {cfg['sigma_floor']:.1f}"
    else:
        typ_noise = float(cfg["sigma"])
        formula = f"abs {cfg['sigma']:.2f}"
        
    ratio = typ_noise / m_std
    feature_stds.append({
        "feature": feat,
        "within_crop_std_mean": round(m_std, 3),
        "within_crop_std_min": round(min_std, 3),
        "within_crop_std_max": round(max_std, 3),
        "noise_1sigma_typical": round(typ_noise, 3),
        "noise_to_within_std_ratio": round(ratio, 3),
        "noise_larger_than_natural_std": (ratio > 1.0)
    })

df_stds = pd.DataFrame(feature_stds)
print("=== FEATURE DISPERSION VS APPLIED NOISE ===")
print(df_stds.to_string(index=False))

# 2. Noise-by-Feature Stage 2 Refusal Test (Seed 1001)
rng_1001 = np.random.default_rng(seed=1001)
noise_matrix_1001 = apply_noise(X_val, profile="primary_1sigma", seed=1001) - X_val

single_feat_results = []
# Clean
flags_clean = [engine.assess_ood({f: float(X_val[i, j]) for j, f in enumerate(CANONICAL_FEATURES)}, risk="balanced")["flagged"] for i in range(330)]
single_feat_results.append({"condition": "CLEAN_ZERO_NOISE", "refused_balanced": sum(flags_clean), "pct": round(sum(flags_clean)/330*100, 2)})

for col_idx, feat in enumerate(CANONICAL_FEATURES):
    X_single = X_val.copy()
    X_single[:, col_idx] += noise_matrix_1001[:, col_idx]
    flags = [engine.assess_ood({f: float(X_single[i, j]) for j, f in enumerate(CANONICAL_FEATURES)}, risk="balanced")["flagged"] for i in range(330)]
    single_feat_results.append({"condition": f"{feat}_alone", "refused_balanced": sum(flags), "pct": round(sum(flags)/330*100, 2)})

flags_all = [engine.assess_ood({f: float((X_val + noise_matrix_1001)[i, j]) for j, f in enumerate(CANONICAL_FEATURES)}, risk="balanced")["flagged"] for i in range(330)]
single_feat_results.append({"condition": "ALL_FEATURES_NOISY", "refused_balanced": sum(flags_all), "pct": round(sum(flags_all)/330*100, 2)})

print("\n=== SINGLE-FEATURE NOISE REFUSAL TEST (Seed 1001, Balanced) ===")
print(pd.DataFrame(single_feat_results).to_string(index=False))

# 3. Refusal Rate per Crop under Full Noise (Balanced)
crop_refusal = {}
for crop in classes:
    crop_refusal[crop] = {"refused": 0, "total": 0}

for ds in eval_noisy_list:
    y_t = ds["y_true"]
    for i in range(len(ds["X_arr"])):
        c = y_t[i]
        crop_refusal[c]["total"] += 1
        if ds["stage2_results"][i]["min_distance"] > 12.92:
            crop_refusal[c]["refused"] += 1

crop_table = []
for c, st in crop_refusal.items():
    rate = st["refused"] / st["total"] * 100
    crop_table.append({"crop": c, "refused": st["refused"], "total": st["total"], "refusal_rate_pct": round(rate, 2)})

df_crop = pd.DataFrame(crop_table).sort_values(by="refusal_rate_pct", ascending=False)
print("\n=== PER-CROP REFUSAL RATE UNDER FULL NOISE (Balanced, N=75/crop) ===")
print(df_crop.head(6).to_string(index=False))
print("...")
print(df_crop.tail(4).to_string(index=False))


=== FEATURE DISPERSION VS APPLIED NOISE ===
    feature  within_crop_std_mean  within_crop_std_min  within_crop_std_max  noise_1sigma_typical  noise_to_within_std_ratio  noise_larger_than_natural_std
          N                11.930               10.929               12.840                 5.046                      0.423                          False
          P                 7.583                6.667                8.455                 5.329                      0.703                          False
          K                 3.140                2.661                3.505                 4.813                      1.533                           True
temperature                 2.779                0.809                9.582                 1.500                      0.540                          False
   humidity                 3.318                1.119               10.705                 5.000                      1.507                           True
         ph         


=== SINGLE-FEATURE NOISE REFUSAL TEST (Seed 1001, Balanced) ===
         condition  refused_balanced   pct
  CLEAN_ZERO_NOISE                 4  1.21
           N_alone                 5  1.52
           P_alone                12  3.64
           K_alone                21  6.36
 temperature_alone                23  6.97
    humidity_alone                38 11.52
          ph_alone                18  5.45
    rainfall_alone                22  6.67
ALL_FEATURES_NOISY               122 36.97

=== PER-CROP REFUSAL RATE UNDER FULL NOISE (Balanced, N=75/crop) ===
      crop  refused  total  refusal_rate_pct
 muskmelon       62     75             82.67
     apple       59     75             78.67
    grapes       59     75             78.67
  chickpea       52     75             69.33
    banana       36     75             48.00
watermelon       36     75             48.00
...
      crop  refused  total  refusal_rate_pct
      jute       16     75             21.33
    lentil        5     75

---
## Section D: Cross-Check Unknown Detection Rates

We evaluate detection across all five empirical reference cutoffs on independent evaluation seeds (Box `seed=202`, Shuffled `seed=404`, $N = 1,000$ each).

In [8]:
box_202 = generate_box_uniform(1000, train_min, train_max, seed=202)
shuf_404 = generate_shuffled_columns(X_val_eval, 1000, seed=404)

res_box_202 = evaluate_single_dataset("box_202", box_202, np.full(1000, "UNK"), engine)
res_shuf_404 = evaluate_single_dataset("shuf_404", shuf_404, np.full(1000, "UNK"), engine)

d2_box_202 = np.array([r["min_distance"] for r in res_box_202["stage2_results"]])
d2_shuf_404 = np.array([r["min_distance"] for r in res_shuf_404["stage2_results"]])

cutoffs = [9.53, 10.78, 11.71, 12.92, 13.75]
cc_records = []
for c in cutoffs:
    b_det = float(np.mean(d2_box_202 > c) * 100.0)
    s_det = float(np.mean(d2_shuf_404 > c) * 100.0)
    cc_records.append({
        "cutoff_d2": c,
        "box_detection_pct": round(b_det, 1),
        "shuffled_detection_pct": round(s_det, 1)
    })

print("=== STAGE 2 DETECTION ACROSS 5 CUTOFFS (Seeds 202 & 404, N=1000 each) ===")
print(pd.DataFrame(cc_records).to_string(index=False))


=== STAGE 2 DETECTION ACROSS 5 CUTOFFS (Seeds 202 & 404, N=1000 each) ===
 cutoff_d2  box_detection_pct  shuffled_detection_pct
      9.53               99.8                    95.6
     10.78               99.8                    94.2
     11.71               99.8                    92.7
     12.92               99.7                    90.1
     13.75               99.5                    88.0


---
## Section E: Analysis of Slipped Shuffled Unknowns

We analyze the subset of column-shuffled inputs (seed 4040, $N = 1,000$) that slip past Stage 2 and Stage 5 caution checks to receive a "recommend" verdict.

In [9]:
shuf_summary = {}
for p in ["conservative", "balanced", "bold"]:
    v_shuf = compute_profile_verdicts(eval_shuf_main, engine, p)
    rec_indices = [i for i, v in enumerate(v_shuf) if v["verdict"] == "recommend"]
    
    crops = [eval_shuf_main["stage2_results"][i]["nearest_known"][0] for i in rec_indices]
    pcts = [eval_shuf_main["stage2_results"][i]["percentile"] for i in rec_indices]
    probs = [eval_shuf_main["top_predicted_probs"][i] for i in rec_indices]
    stabs = [eval_shuf_main["stabilities"][i] for i in rec_indices]
    
    crop_counts = pd.Series(crops).value_counts().to_dict()
    shuf_summary[p] = {
        "count": len(rec_indices),
        "d2_percentile": {"min": min(pcts), "median": float(np.median(pcts)), "max": max(pcts)},
        "top_probability": {"min": min(probs), "median": float(np.median(probs)), "max": max(probs)},
        "stability": {"min": min(stabs), "median": float(np.median(stabs)), "max": max(stabs)},
        "top_crops": list(crop_counts.items())[:5]
    }
    print(f"[{p.upper()}] Recommended: {len(rec_indices)} / 1000")
    print(f"  d2 percentile range: min={min(pcts):.1f}%, med={np.median(pcts):.1f}%, max={max(pcts):.1f}%")
    print(f"  Top crops: {list(crop_counts.items())[:4]}")


[CONSERVATIVE] Recommended: 28 / 1000
  d2 percentile range: min=4.4%, med=59.7%, max=89.4%
  Top crops: [('pigeonpeas', 9), ('mothbeans', 4), ('maize', 3), ('papaya', 3)]


[BALANCED] Recommended: 45 / 1000
  d2 percentile range: min=4.4%, med=73.5%, max=94.9%
  Top crops: [('pigeonpeas', 14), ('mothbeans', 9), ('maize', 6), ('papaya', 4)]


[BOLD] Recommended: 70 / 1000
  d2 percentile range: min=4.4%, med=80.2%, max=96.9%
  Top crops: [('pigeonpeas', 25), ('mothbeans', 11), ('maize', 9), ('papaya', 5)]


---
## Section F: Paired Bootstrap Differences & Profile Separation

We evaluate whether the trade-off across adjacent profiles is statistically distinct.  
Using $B = 2,000$ paired bootstrap resamples (`seed=7`), we resample the 330 base validation rows with replacement, keeping each base row linked to its 5 noisy copies ($N = 1,650$ evaluated per replicate).  
Note that confidence intervals for risk and coverage represent paired difference distributions.

In [10]:
# Precompute verdicts for all noisy rows
all_noisy_verdicts = {}
for p in ["conservative", "balanced", "bold"]:
    v_p = []
    for ds in eval_noisy_list:
        v_p.extend(compute_profile_verdicts(ds, engine, p))
    all_noisy_verdicts[p] = v_p

all_noisy_trues = []
all_noisy_preds = []
for ds in eval_noisy_list:
    df_x = pd.DataFrame(ds["X_arr"], columns=CANONICAL_FEATURES)
    probs = engine.pipeline.predict_proba(df_x)
    preds = [engine.classes[idx] for idx in np.argmax(probs, axis=1)]
    all_noisy_preds.extend(preds)
    all_noisy_trues.extend(ds["y_true"])

all_noisy_preds = np.array(all_noisy_preds)
all_noisy_trues = np.array(all_noisy_trues)

# Resampling 330 base rows with 5 linked copies
rng_boot = np.random.default_rng(seed=7)
n_base = 330
n_seeds = 5
B = 2000

cov_diff_bal_cons = []
risk_diff_bal_cons = []
cov_diff_bold_bal = []
risk_diff_bold_bal = []

for b in range(B):
    boot_base = rng_boot.choice(n_base, size=n_base, replace=True)
    # Expand to 1650 indices
    boot_indices = []
    for s_idx in range(n_seeds):
        boot_indices.extend(boot_base + s_idx * n_base)
    boot_indices = np.array(boot_indices)
    
    # Metrics per profile
    prof_cov = {}
    prof_risk = {}
    for p in ["conservative", "balanced", "bold"]:
        v_list = all_noisy_verdicts[p]
        rec_flags = np.array([v_list[i]["verdict"] == "recommend" for i in boot_indices])
        n_rec = int(np.sum(rec_flags))
        cov = n_rec / len(boot_indices)
        if n_rec > 0:
            sub_preds = all_noisy_preds[boot_indices][rec_flags]
            sub_trues = all_noisy_trues[boot_indices][rec_flags]
            risk = float(np.mean(sub_preds != sub_trues))
        else:
            risk = 0.0
        prof_cov[p] = cov
        prof_risk[p] = risk
        
    cov_diff_bal_cons.append(prof_cov["balanced"] - prof_cov["conservative"])
    risk_diff_bal_cons.append(prof_risk["balanced"] - prof_risk["conservative"])
    cov_diff_bold_bal.append(prof_cov["bold"] - prof_cov["balanced"])
    risk_diff_bold_bal.append(prof_risk["bold"] - prof_risk["balanced"])

def calc_ci(diff_arr):
    return float(np.mean(diff_arr)), float(np.percentile(diff_arr, 2.5)), float(np.percentile(diff_arr, 97.5))

m_cov_bc, l_cov_bc, u_cov_bc = calc_ci(cov_diff_bal_cons)
m_rk_bc, l_rk_bc, u_rk_bc = calc_ci(risk_diff_bal_cons)

m_cov_bb, l_cov_bb, u_cov_bb = calc_ci(cov_diff_bold_bal)
m_rk_bb, l_rk_bb, u_rk_bb = calc_ci(risk_diff_bold_bal)

print("=== PAIRED BOOTSTRAP DIFFERENCES (B=2000, Seed 7) ===")
print("1. Balanced minus Conservative:")
print(f"  Delta Coverage:      +{m_cov_bc*100:.2f}% (95% CI: [{l_cov_bc*100:.2f}%, {u_cov_bc*100:.2f}%])")
print(f"  Delta Selective Risk: +{m_rk_bc*100:.2f}% (95% CI: [{l_rk_bc*100:.2f}%, {u_rk_bc*100:.2f}%])")

print("\n2. Bold minus Balanced:")
print(f"  Delta Coverage:      +{m_cov_bb*100:.2f}% (95% CI: [{l_cov_bb*100:.2f}%, {u_cov_bb*100:.2f}%])")
print(f"  Delta Selective Risk: +{m_rk_bb*100:.2f}% (95% CI: [{l_rk_bb*100:.2f}%, {u_rk_bb*100:.2f}%])")


=== PAIRED BOOTSTRAP DIFFERENCES (B=2000, Seed 7) ===
1. Balanced minus Conservative:
  Delta Coverage:      +11.15% (95% CI: [9.52%, 12.85%])
  Delta Selective Risk: +1.09% (95% CI: [0.05%, 2.45%])

2. Bold minus Balanced:
  Delta Coverage:      +9.51% (95% CI: [7.94%, 11.21%])
  Delta Selective Risk: +1.41% (95% CI: [0.49%, 2.57%])


---
## Section G: Summary Artifact Export

We export `docs/figures/risk_coverage_summary.json` containing key selective classification findings.

In [11]:
summary_artifact = {
    "experiment": "E7_selective_classification",
    "updated_at": "2026-11-08",
    "role": "Role 2: Trust Engine Engineer",
    "eval_model": "crop_pipeline_calibrated.joblib",
    "operating_points": {
        "clean_validation": {
            "conservative": {"coverage": 251/330, "selective_risk": 0.0, "recommended": 251, "wrong": 0},
            "balanced": {"coverage": 288/330, "selective_risk": 0.0, "recommended": 288, "wrong": 0},
            "bold": {"coverage": 315/330, "selective_risk": 0.0, "recommended": 315, "wrong": 0}
        },
        "noisy_validation_pooled": {
            "conservative": {"coverage": 517/1650, "selective_risk": 4/517, "recommended": 517, "wrong": 4},
            "balanced": {"coverage": 701/1650, "selective_risk": 13/701, "recommended": 701, "wrong": 13},
            "bold": {"coverage": 858/1650, "selective_risk": 28/858, "recommended": 858, "wrong": 28}
        }
    },
    "paired_bootstrap_differences": {
        "balanced_minus_conservative": {
            "mean_coverage_diff": round(m_cov_bc, 4),
            "coverage_diff_ci_95": [round(l_cov_bc, 4), round(u_cov_bc, 4)],
            "mean_selective_risk_diff": round(m_rk_bc, 4),
            "selective_risk_diff_ci_95": [round(l_rk_bc, 4), round(u_rk_bc, 4)]
        },
        "bold_minus_balanced": {
            "mean_coverage_diff": round(m_cov_bb, 4),
            "coverage_diff_ci_95": [round(l_cov_bb, 4), round(u_cov_bb, 4)],
            "mean_selective_risk_diff": round(m_rk_bb, 4),
            "selective_risk_diff_ci_95": [round(l_rk_bb, 4), round(u_rk_bb, 4)]
        }
    },
    "manufactured_unknowns_refusal_rates": {
        "box_uniform_seed_2020": {"conservative": 99.6, "balanced": 99.5, "bold": 99.5},
        "shuffled_column_seed_4040": {"conservative": 91.1, "balanced": 88.5, "bold": 86.7}
    }
}

summary_out_path = FIGURES_DIR / "risk_coverage_summary.json"
with open(summary_out_path, "w", encoding="utf-8") as f:
    json.dump(summary_artifact, f, indent=2)

print(f"Successfully exported E7 summary artifact to: {summary_out_path}")


Successfully exported E7 summary artifact to: C:\Users\varsh\OneDrive\Desktop\crop-trust-engine\docs\figures\risk_coverage_summary.json
